In [6]:
from pulp import *
import pandas as pd

# masses in dalton
phosphorus = 30.973761998
oxygen = 31.989829239
d_ribose = 150.05282342

tail_5prime = [d_ribose, oxygen, 2*oxygen + phosphorus, oxygen]

base_masses = pd.read_csv("../masses_all.tsv", sep="\t").set_index("nucleoside", drop=False)
# dbg:
# masses = masses.loc[["A", "C", "G", "U"]]

In [7]:
base_masses

,nucleoside,monoisotopic_mass
nucleoside,,
A,A,267.09675
G,G,283.09167
C,C,243.08552
U,U,244.06954
1A,1A,281.11240
...,...,...
10G,10G,409.15970
102G,102G,425.15470
105G,105G,538.20230


In [8]:
import random, re
from scipy.stats import norm

nucleoside_re = re.compile(r"\d*[ACGU]")

true_sequence = nucleoside_re.findall("CCUGUU")
print(true_sequence)
n_fragments = 20

# sample random fragments from true sequence
fragment_noise = norm.rvs(scale=0.5, size=n_fragments)
def random_fragment():
    l = random.randint(0, len(true_sequence) - 1)
    r = random.randint(l + 1, len(true_sequence))
    return l, r
fragments = [random_fragment() for _ in range(n_fragments)]
fragment_masses = [max(sum(base_masses.loc[b, "monoisotopic_mass"] for b in true_sequence[l:r]) + noise, 0.0) for noise, (l, r) in zip(fragment_noise, fragments)]

start_fragments = [i for i in range(n_fragments) if fragments[i][0] == 0]
end_fragments = [i for i in range(n_fragments) if fragments[i][1] == len(true_sequence)]

fragments, fragment_masses, fragment_noise

['C', 'C', 'U', 'G', 'U', 'U']


([(1, 4),
  (0, 4),
  (0, 1),
  (0, 6),
  (3, 5),
  (0, 5),
  (5, 6),
  (2, 4),
  (5, 6),
  (0, 2),
  (3, 6),
  (4, 6),
  (4, 6),
  (4, 6),
  (5, 6),
  (1, 2),
  (1, 3),
  (3, 6),
  (3, 6),
  (3, 6)],
 [np.float64(769.7298247901017),
  np.float64(1013.7632974093988),
  np.float64(242.7171421918622),
  np.float64(1500.3451048490501),
  np.float64(527.991827252681),
  np.float64(1257.759411757136),
  np.float64(244.47005139425818),
  np.float64(526.7598853905989),
  np.float64(244.16708334889674),
  np.float64(486.6089630871313),
  np.float64(770.9778076529692),
  np.float64(488.69458594354506),
  np.float64(487.2668792560211),
  np.float64(488.50912314460015),
  np.float64(243.81036741387524),
  np.float64(243.44476915622124),
  np.float64(487.4413625336665),
  np.float64(770.7373739793899),
  np.float64(771.5945648002995),
  np.float64(771.2534833753406)],
 array([-0.51690521,  0.43104741, -0.36837781, -1.12622515,  0.83061725,
         0.35762176,  0.40051139, -0.40132461,  0.09754335

In [56]:
def reduce_alphabet(base_masses):
    start_masses = pd.Series(sorted(fragment_masses[i] for i in start_fragments))
    end_masses = pd.Series(sorted(fragment_masses[i] for i in end_fragments))

    print(start_masses)
    print(end_masses)


    def mass_diffs(masses):
        diffs = set(masses[1:].values - masses[:-1].values)
        print(diffs)
        return diffs
    
    diffs = sorted(mass_diffs(start_masses) | mass_diffs(end_masses))

    def is_similar(mass_a, mass_b):
        print(mass_a, mass_b, abs(mass_a - mass_b))
        return abs(mass_a - mass_b) < 1.0

    def get_similar_nucleosides():
        i_diff = 0
        sorted_masses = base_masses.sort_values("monoisotopic_mass")
        for item in sorted_masses.itertuples():
            # move until close enough to next mass in base_masses
            # (diffs are sorted)
            while i_diff < len(diffs) and diffs[i_diff] < item.monoisotopic_mass - 1.0:
                i_diff += 1
            j = 0
            while i_diff + j < len(diffs) and diffs[i_diff + j] <= item.monoisotopic_mass + 1.0:
                diff_mass = diffs[i_diff + j]
                if is_similar(diff_mass, item.monoisotopic_mass):
                    yield item.nucleoside
                j += 1

    return base_masses.loc[list(set(get_similar_nucleosides()))]

In [57]:
reduce_alphabet(base_masses)

0     242.717142
1     486.608963
2    1013.763297
3    1257.759412
4    1500.345105
dtype: float64
0      243.810367
1      244.167083
2      244.470051
3      487.266879
4      488.509123
5      488.694586
6      770.737374
7      770.977808
8      771.253483
9      771.594565
10    1500.345105
dtype: float64
{np.float64(242.5856930919142), np.float64(243.89182089526912), np.float64(243.99611434773715), np.float64(527.1543343222675)}
{np.float64(0.3567159350214979), np.float64(0.30296804536143895), np.float64(0.18546279894491136), np.float64(1.2422438885790257), np.float64(0.2404336735793322), np.float64(0.27567572237137483), np.float64(0.34108142495892935), np.float64(242.79682786176295), np.float64(728.7505400487506), np.float64(282.0427880358448)}
           nucleoside  monoisotopic_mass
nucleoside                              
C                   C          243.08552
U                   U          244.06954
A                   A          267.09675
1A                 1A          2

,nucleoside,monoisotopic_mass
nucleoside,,
0A,0A,281.11240
8A,8A,281.11240
6A,6A,281.11240
09A,09A,282.09640
19A,19A,282.09640
2A,2A,281.11240
1A,1A,281.11240
C,C,243.08552
U,U,244.06954


In [216]:
def estimate_seq(base_masses):
    prob = LpProblem("RNA sequencing", LpMinimize)
    # i = 1,...,n: positions in the sequence
    # j = 1,...,m: fragments
    # b = 1,...,k: (modified) bases

    n = len(true_sequence)
    m = len(fragment_masses)
    bases = base_masses.index.values

    # x: binary variables indicating fragment j presence at position i
    x = [[LpVariable(f"x_{i},{j}", lowBound=0, upBound=1, cat=LpInteger) for j in range(m)] for i in range(n)]
    # y: binary variables indicating base at position i
    y = [{b: LpVariable(f"y_{i},{b}", lowBound=0, upBound=1, cat=LpInteger) for b in bases} for i in range(n)]
    # z: binary variables indicating product of x and y
    z = [[{b: LpVariable(f"z_{i},{j},{b}", lowBound=0, upBound=1, cat=LpInteger) for b in bases} for j in range(m)] for i in range(n)]
    # weight_diff_abs: absolute value of weight_diff
    weight_diff_abs = [LpVariable(f"weight_diff_abs_{j}", lowBound=0, cat=LpContinuous) for j in range(m)]
    # weight_diff: difference between fragment monoisotopic mass and sum of masses of bases in fragment as estimated in the MILP
    weight_diff = [fragment_masses[j] - lpSum([z[i][j][b] * base_masses.loc[b, "monoisotopic_mass"] for i in range(n) for b in bases]) for j in range(m)]

    # optimization function
    prob += lpSum([weight_diff_abs[j] for j in range(m)])

    # select one base per position
    for i in range(n):
        
        prob += lpSum([y[i][b] for b in bases]) == 1

    # fill z with the product of binary variables x and y
    for i in range(n):
        for j in range(m):
            for b in bases:
                prob += z[i][j][b] <= x[i][j]
                prob += z[i][j][b] <= y[i][b]
                prob += z[i][j][b] >= x[i][j] + y[i][b] - 1

    # ensure that fragment is aligned continuously
    # (no gaps: if x[i,j] = 1 and x[i+2,j] = 1, then x[i+1,j] = 1)
    for j in range(m):
        for i in range(n - 2):
            prob += x[i][j] + x[i + 2][j] - 1 <= x[i + 1][j]

    # ensure that start fragments are aligned at the beginning of the sequence
    for j in start_fragments:
        x[0][j].setInitialValue(1)
        x[0][j].fixValue()

    # ensure that end fragments are aligned at the end of the sequence
    for j in end_fragments:
        x[n - 1][j].setInitialValue(1)
        x[n - 1][j].fixValue()

    # constrain weight_diff_abs to be the absolute value of weight_diff
    for j in range(m):
        prob += weight_diff_abs[j] >= weight_diff[j]
        prob += weight_diff_abs[j] >= -weight_diff[j]
    
    import pulp
    gurobi = pulp.getSolver("GUROBI_CMD")
    gurobi.msg = False
    result_ = prob.solve(gurobi)

    def get_base(i):
        for b in bases:
            if y[i][b].value() == 1:
                return b
        return None

    return [get_base(i) for i in range(n)]
    

In [217]:
from collections import defaultdict
from sklearn.cluster import KMeans
import numpy as np

reduce_to_k = 6

def reduce_alphabet(masses):
    kmeans = KMeans(n_clusters=reduce_to_k, random_state=213126)
    kmeans.fit(masses["monoisotopic_mass"].values.reshape(-1, 1))
    pseudo_nucs = [f"n{k}" for k in range(reduce_to_k)]
    nuc_mapping = defaultdict(list)
    for nuc, label in zip(masses["nucleoside"], kmeans.labels_):
        nuc_mapping[pseudo_nucs[label]].append(nuc)
    return (
        pd.DataFrame({
            "nucleoside": pseudo_nucs,
            "monoisotopic_mass": kmeans.cluster_centers_[:, 0]
        }).set_index("nucleoside", drop=False),
        nuc_mapping
    )

In [218]:
def expand_alphabet(seq, nuc_mapping):
    return {nuc for pseudo_nuc in seq for nuc in nuc_mapping[pseudo_nuc]}

In [219]:
start_fragments, end_fragments

([0, 3, 15], [1, 9, 12, 13, 15, 18, 19])

In [220]:
current_masses = base_masses
seq = None
last_masses_count = None

while True:
    print(f"remaining masses: {current_masses}")
    
    if current_masses.shape[0] > reduce_to_k and current_masses.shape[0] != last_masses_count:
        pseudo_masses, nuc_mapping = reduce_alphabet(current_masses)
        seq = estimate_seq(pseudo_masses)
        print(seq, nuc_mapping)
    else:
        seq = estimate_seq(current_masses)
        break

    last_masses_count = current_masses.shape[0]
    current_masses = current_masses.loc[list(expand_alphabet(seq, nuc_mapping))]
seq

remaining masses:            nucleoside  monoisotopic_mass
nucleoside                              
A                   A          267.09675
G                   G          283.09167
C                   C          243.08552
U                   U          244.06954
1A                 1A          281.11240
...               ...                ...
10G               10G          409.15970
102G             102G          425.15470
105G             105G          538.20230
104G             104G          571.21260
106G             106G          571.21260

[67 rows x 2 columns]


/home/johannes/micromamba/envs/pystats/lib/python3.12/site-packages/pulp/pulp.py:1316: UserWarning: Spaces are not permitted in the name. Converted to '_'
  warnings.warn("Spaces are not permitted in the name. Converted to '_'")


['n4', 'n5', 'n1', 'n0', 'n3', 'n4'] defaultdict(<class 'list'>, {'n4': ['A', 'G', 'C', 'U', '1A', '2A', '0A', '8A', '6A', '19A', '09A', '67A', '01A', '28A', '06A', '66A', '019A', '68A', '1G', '0G', '2G', '7G'], 'n0': ['64A', '066A', '621A', '61A', '60A', '01G', '02G', '22G', '27G', '4G', '022G', '027G', '227G', '42G', '34G', '342G', '100G', '101G', '103G'], 'n3': ['65A', '2161A', '69A', '2160A', '62A', '63A', '662A', '2165A', '2164A', '47G', '347G', '10G', '102G'], 'n1': ['2162A', '2163A', '00A', '348G', '3470G', '3480G', '00G'], 'n5': ['3483G', '34830G', '34832G', '105G'], 'n2': ['104G', '106G']})
remaining masses:            nucleoside  monoisotopic_mass
nucleoside                              
A                   A          267.09675
101G             101G          311.12300
22G               22G          311.12300
01A               01A          295.12810
8A                 8A          281.11240
...               ...                ...
64A               64A          309.10730
G     

/home/johannes/micromamba/envs/pystats/lib/python3.12/site-packages/pulp/pulp.py:1316: UserWarning: Spaces are not permitted in the name. Converted to '_'
  warnings.warn("Spaces are not permitted in the name. Converted to '_'")


['n3', 'n2', 'n5', 'n0', 'n1', 'n3'] defaultdict(<class 'list'>, {'n3': ['A', '01A', '8A', '6A', '0G', '67A', '019A', 'C', '1A', '68A', '09A', '2G', 'U', '06A', '66A', '1G', '28A', '7G', '19A', '0A', 'G', '2A'], 'n0': ['101G', '22G', '01G', '227G', '066A', '02G', '42G', '022G', '103G', '4G', '27G', '027G', '621A', '34G', '64A', '100G', '61A'], 'n1': ['662A', '63A', '347G', '62A', '2165A', '10G', '102G', '47G', '2160A'], 'n2': ['34830G', '105G', '3483G', '34832G', '00G'], 'n5': ['00A', '2164A', '3480G', '2162A', '2163A', '3470G', '348G'], 'n4': ['65A', '60A', '2161A', '69A', '342G']})
remaining masses:            nucleoside  monoisotopic_mass
nucleoside                              
A                   A          267.09675
22G               22G          311.12300
01A               01A          295.12810
8A                 8A          281.11240
662A             662A          426.14990
34830G         34830G          524.18670
00A               00A          479.10530
2164A           2164A 

...

['n4', 'n4', 'n3', 'n4', 'n4', 'n4'] defaultdict(<class 'list'>, {'n3': ['A', '01A', '8A', '6A', '0G', '67A', '019A', '1A', '68A', '09A', '2G', '06A', '66A', '1G', '28A', '7G', '19A', '0A', 'G', '2A'], 'n0': ['22G', '01G', '227G', '066A', '02G', '42G', '022G', '103G', '4G', '27G', '027G', '621A', '34G', '100G', '64A', '101G', '61A'], 'n1': ['662A', '2164A', '63A', '347G', '62A', '2165A', '10G', '102G', '47G', '2160A'], 'n2': ['34830G', '105G', '3483G', '34832G', '00G'], 'n5': ['00A', '3480G', '2162A', '2163A', '3470G', '348G'], 'n4': ['C', 'U']})
remaining masses:            nucleoside  monoisotopic_mass
nucleoside                              
A                   A          267.09675
01A               01A          295.12810
8A                 8A          281.11240
6A                 6A          281.11240
0G                 0G          297.10730
67A               67A          295.09170
019A             019A          296.11210
C                   C          243.08552
1A                 

/home/johannes/micromamba/envs/pystats/lib/python3.12/site-packages/pulp/pulp.py:1316: UserWarning: Spaces are not permitted in the name. Converted to '_'
  warnings.warn("Spaces are not permitted in the name. Converted to '_'")


['n1', None, 'n1', 'n1', 'n1', 'n1'] defaultdict(<class 'list'>, {'n3': ['A'], 'n0': ['01A', '67A', '019A', '06A', '66A', '28A'], 'n2': ['8A', '6A', '1A', '09A', '19A', '0A', 'G', '2A'], 'n5': ['0G', '68A', '2G', '1G'], 'n1': ['C', 'U'], 'n4': ['7G']})
remaining masses:            nucleoside  monoisotopic_mass
nucleoside                              
U                   U          244.06954
C                   C          243.08552


/home/johannes/micromamba/envs/pystats/lib/python3.12/site-packages/pulp/pulp.py:1316: UserWarning: Spaces are not permitted in the name. Converted to '_'
  warnings.warn("Spaces are not permitted in the name. Converted to '_'")


['C', 'U', 'U', 'U', 'U', 'U']